# Chapter 10: Virtual Graphs and Mappings

**Level 3: Integration & Engineering** · ⏱ about 3 hours · Dataset: **EduGraph** (new)

### What you will learn

1. The data integration problem, and Stardog's two answers: **materialize** (import) and **virtualize** (query in place)
2. **Data sources**, **virtual graphs** and **mappings**: how Stardog reads data that isn't RDF
3. The **SMS2** mapping language: `MAPPING … FROM … TO … WHERE`, IRI templates, type casts (and how R2RML compares)
4. **Hands-on:** map a school information system's CSV exports into the graph, and query them **together with** a hand-written curriculum
5. **Prerequisite gaps:** a question neither source can answer alone
6. **Live virtual graphs** over a relational database, query **pushdown**, and when to virtualize vs materialize

### About this chapter's setup

Stardog Cloud runs in the cloud, so it **can't reach a database on your laptop**. The hands-on therefore **imports CSV files through mappings**, which works on any plan. Section 10.7 shows the **live virtual graph** path for a database reachable from the internet. It runs only if you configure one in `.env`, and otherwise just explains.

In [1]:
import sys
from pathlib import Path

SRC = str(Path('..', 'src').resolve())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pandas as pd
import stardog
from rdflib import URIRef

from kg import client, imports, reasoning, sparql

pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)
conn = client.connect()
PREFIX = 'urn:tutorial:ch10'
ROOT = Path('..')
EDU = ROOT / 'data' / 'edugraph'
MAPPINGS = ROOT / 'ontology' / 'mappings'

CURRICULUM = f'{PREFIX}:curriculum'
SIS = {name: f'{PREFIX}:sis:{name}' for name in ('students', 'enrollments', 'assessments')}
ALL = [CURRICULUM, *SIS.values()]

## 10.1 The integration problem

Real knowledge lives in many systems: a school has a **curriculum** (topics, prerequisites, courses) designed by teachers, and a **school information system** (SIS) with students, enrollments and grades. Questions worth asking need **both**:

> *"Which students are enrolled in a course whose topics depend on something they failed?"*

Stardog offers two ways to bring the SIS in:

| | **Materialize** (import) | **Virtualize** (query in place) |
|---|---|---|
| How | Map the source to RDF **once**, store the triples in the graph | Map the source to RDF **at query time**; the data stays in the source |
| Freshness | As fresh as the last import | Always current |
| Speed | Fast: local indexes | Depends on the source and the network |
| Source load | Only during the import | On every query |
| Reasoning, full-text, etc. | Everything works | Most things work, pushed down to the source where possible |
| Works for | Files (CSV, JSON), any source | Databases and APIs reachable **from the Stardog server** |

Both use the **same mapping**. Only the way you run it differs.

## 10.2 Data sources, virtual graphs and mappings

```
  source system                       Stardog
 ┌──────────────┐   data source   ┌───────────────────────────────────────────────┐
 │ PostgreSQL,  │◄────────────────│  virtual graph = data source + mapping         │
 │ MySQL, …     │   (connection)  │  query:  GRAPH <virtual://sis> { … }           │
 └──────────────┘                 │                                                │
 ┌──────────────┐   import        │  named graph <urn:…:students>                  │
 │ students.csv │────────────────►│  (mapping applied once, triples stored)        │
 └──────────────┘   + mapping     └───────────────────────────────────────────────┘
```

- A **data source** is a saved connection (JDBC URL, user, password, options).
- A **mapping** says how rows become triples.
- A **virtual graph** = data source + mapping, queried as `GRAPH <virtual://name> { … }`.
- An **import** runs a mapping over a file (or a virtual graph) once and stores the result in a named graph.

> 🔁 **Neo4j equivalent:** `LOAD CSV` / APOC procedures import data (materialize). Neo4j has no general virtual-graph layer; its closest feature, Fabric, federates Neo4j databases only.

## 10.3 The SMS2 mapping language

SMS2 (Stardog Mapping Syntax 2) reads like SPARQL:

```sparql
MAPPING <urn:tutorial:mapping:students>        # a name for the mapping
FROM CSV { }                                   # where the rows come from: CSV, JSON, SQL, …
TO {                                           # a triple template, filled in for each row
  ?student a edu:Student ;
      edu:name ?full_name ;
      edu:year ?year_number .
}
WHERE {                                        # compute template variables from the row's columns
  BIND(TEMPLATE("http://example.org/edu/student/{student_id}") AS ?student)
  BIND(CONCAT(?first_name, " ", ?last_name) AS ?full_name)
  BIND(xsd:integer(?year) AS ?year_number)
}
```

- Each **column** is available as a variable: `?student_id`, `?first_name`, …
- **`TEMPLATE`** builds IRIs from column values. This is the most important design decision: the template must produce **the same IRIs** as the rest of the graph, or nothing will join.
- CSV values are strings. **Cast** them (`xsd:integer`, `xsd:date`) so numbers and dates behave (Chapter 3).
- Only the `FROM` clause changes between sources: `FROM CSV {}`, `FROM JSON {…}`, `FROM SQL { SELECT … }`.

**R2RML**, the W3C standard, expresses the same thing in (much more verbose) Turtle. Stardog accepts both; SMS2 is easier to read and write.

```turtle
<#Students> rr:logicalTable [ rr:tableName "students" ] ;
  rr:subjectMap [ rr:template "http://example.org/edu/student/{student_id}" ; rr:class edu:Student ] ;
  rr:predicateObjectMap [ rr:predicate edu:studentId ; rr:objectMap [ rr:column "student_id" ] ] .
```

## 10.4 Hands-on: import the school information system

### The curriculum: hand-written Turtle

21 topics with prerequisites (`edu:requires`) and 4 courses that cover them. Teachers maintain this directly as RDF:

In [2]:
print((EDU / 'curriculum.ttl').read_text(encoding='utf-8')[:1600], '\n…')

with client.transaction(conn) as stats:
    conn.clear(graph_uri=CURRICULUM)
    conn.add(stardog.content.File(str(EDU / 'curriculum.ttl')), graph_uri=CURRICULUM)
print('curriculum:', stats)

# EduGraph curriculum (Chapters 10+): topics, their prerequisites, and courses.
@prefix edu:    <http://example.org/edu#> .
@prefix topic:  <http://example.org/edu/topic/> .
@prefix course: <http://example.org/edu/course/> .
@prefix xsd:    <http://www.w3.org/2001/XMLSchema#> .

# Topics: level 1 = beginner, 3 = advanced. edu:requires = direct prerequisite.
topic:python-basics      a edu:Topic ; edu:name "Python basics" ;        edu:subject "Programming" ;      edu:level 1 .
topic:control-flow       a edu:Topic ; edu:name "Control flow" ;         edu:subject "Programming" ;      edu:level 1 ; edu:requires topic:python-basics .
topic:functions          a edu:Topic ; edu:name "Functions" ;            edu:subject "Programming" ;      edu:level 1 ; edu:requires topic:control-flow .
topic:data-structures    a edu:Topic ; edu:name "Data structures" ;      edu:subject "Programming" ;      edu:level 2 ; edu:requires topic:functions .
topic:numpy              a edu:Topic ; edu:name "NumPy" ;   

curriculum: {'added': 141, 'removed': 0}


### The SIS exports: CSV

Three CSV files, as a school information system would export them (generated by `data/generators/edugraph_sis.py`):

In [3]:
for name in SIS:
    df = pd.read_csv(EDU / 'sis' / f'{name}.csv')
    print(f'{name}.csv: {len(df)} rows')
    display(df.head(3))

students.csv: 40 rows


,student_id,first_name,last_name,email,city,year,enrolled_on
0,S001,Tom,Okafor,tom.s001@student.example.edu,Lagos,1,2023-09-01
1,S002,Hana,Novak,hana.s002@student.example.edu,Lagos,1,2025-08-31
2,S003,Dara,Jensen,dara.s003@student.example.edu,Berlin,1,2023-09-01


enrollments.csv: 77 rows


,student_id,course_code,term
0,S001,DS101,2025-S1
1,S002,MA120,2025-S1
2,S003,DB150,2025-S2


assessments.csv: 340 rows


,assessment_id,student_id,topic,score,taken_on
0,A0001,S001,python-basics,68,2025-02-23
1,A0002,S001,control-flow,79,2025-03-09
2,A0003,S001,functions,71,2025-03-23


The CSVs use **codes** (`S001`, `DS101`, `python-basics`), not IRIs. The mappings turn codes into IRIs with templates that match the curriculum: topic `numpy` becomes `http://example.org/edu/topic/numpy`, exactly the IRI `curriculum.ttl` uses.

### The mappings

One `.sms` file per CSV, in `ontology/mappings/`:

In [4]:
for name in SIS:
    print(f'=== {name}.sms ===')
    print((MAPPINGS / f'{name}.sms').read_text(encoding='utf-8'))

=== students.sms ===
# SMS2 mapping: school information system export students.csv -> RDF (Chapter 10).
# Columns: student_id, first_name, last_name, email, city, year, enrolled_on
PREFIX edu: <http://example.org/edu#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

MAPPING <urn:tutorial:mapping:students>
FROM CSV { }
TO {
  ?student a edu:Student ;
      edu:studentId ?student_id ;
      edu:name ?full_name ;
      edu:email ?email ;
      edu:city ?city ;
      edu:year ?year_number ;
      edu:enrolledOn ?enrolled_date .
}
WHERE {
  BIND(TEMPLATE("http://example.org/edu/student/{student_id}") AS ?student)
  BIND(CONCAT(?first_name, " ", ?last_name) AS ?full_name)
  BIND(xsd:integer(?year) AS ?year_number)
  BIND(xsd:date(?enrolled_on) AS ?enrolled_date)
}

=== enrollments.sms ===
# SMS2 mapping: enrollments.csv -> RDF (Chapter 10).
# Columns: student_id, course_code, term
PREFIX edu: <http://example.org/edu#>

MAPPING <urn:tutorial:mapping:enrollments>
FROM CSV { }
TO {
  ?student 

`enrollments.sms` creates **two** things per row: a direct link `student edu:enrolledIn course` (easy to query), and an `edu:Enrollment` resource with the term (the details), which is the same modelling pattern as the recipes' ingredient lines.

### Run the imports

`kg.imports.import_file()` wraps pystardog's `admin.import_file()`: it uploads the CSV and the mapping, Stardog applies the mapping and stores the triples in the named graph. It clears the graph first, so re-running doesn't mix old and new rows.

In [5]:
print(Path(imports.__file__).read_text())

"""Map CSV / JSON files to RDF with SMS2 mappings and import them into a named graph (Chapter 10)."""
from pathlib import Path

from stardog.content import ImportFile, MappingFile

from kg import client


def import_file(data_file: str | Path, mapping_file: str | Path, graph: str, replace: bool = True) -> int:
  """Import one CSV/JSON file through an SMS2 mapping into `graph`. Returns the graph's triple count.

  replace=True clears the graph first, so re-running an import doesn't mix old and new rows.
  """
  settings = client.load_settings()
  with client.connect(settings=settings) as conn:
    if replace:
      with client.transaction(conn):
        conn.clear(graph_uri=graph)
  with client.admin(settings) as admin:
    admin.import_file(settings.database, MappingFile(str(mapping_file), 'SMS2'), ImportFile(str(data_file)),
                      named_graph=graph)
  with client.connect(settings=settings) as conn:
    result = conn.select(f'SELECT (COUNT(*) AS ?n) {{ GRAPH <{graph}> {

In [6]:
for name, graph in SIS.items():
    triples = imports.import_file(EDU / 'sis' / f'{name}.csv', MAPPINGS / f'{name}.sms', graph)
    print(f'{name:12} -> {graph:40} {triples:5} triples')

students     -> urn:tutorial:ch10:sis:students             280 triples


enrollments  -> urn:tutorial:ch10:sis:enrollments          385 triples


assessments  -> urn:tutorial:ch10:sis:assessments         1700 triples


40 students × 7 properties = 280 triples, 77 enrollments × 5 = 385, and 340 assessments × 5 = 1,700. The casts worked, so scores are numbers and dates are dates:

In [7]:
df = sparql.run_query(conn, '''PREFIX edu: <http://example.org/edu#>
SELECT ?student ?name ?year ?enrolled { ?student edu:name ?name ; edu:year ?year ; edu:enrolledOn ?enrolled } ORDER BY ?student LIMIT 3''',
    graphs=SIS['students'])
print(df.dtypes.to_dict())
df

{'student': <StringDtype(storage='python', na_value=nan)>, 'name': <StringDtype(storage='python', na_value=nan)>, 'year': dtype('int64'), 'enrolled': dtype('O')}


,student,name,year,enrolled
0,http://example.org/edu/student/S001,Tom Okafor,1,2023-09-01
1,http://example.org/edu/student/S002,Hana Novak,1,2025-08-31
2,http://example.org/edu/student/S003,Dara Jensen,1,2023-09-01


## 10.5 Querying across sources

The SIS graphs and the curriculum graph are separate, but their IRIs line up, so a query over all four graphs joins them naturally. The queries are `.rq` files in `src/kg/queries/edugraph/`.

### Course performance

Enrollments (SIS) + course topics (curriculum) + assessments (SIS):

In [8]:
print(sparql.load_query('edugraph/course_performance'))
sparql.run_query(conn, 'edugraph/course_performance', graphs=ALL)

# Per course: enrolled students, assessments taken on the course's topics, and the average score.
# Parameters: none
PREFIX edu: <http://example.org/edu#>

SELECT ?course ?title ?students ?assessments ?avg_score
WHERE {
  ?c a edu:Course ; edu:code ?course ; edu:title ?title .
  { SELECT ?c (COUNT(DISTINCT ?s) AS ?students) WHERE { ?s edu:enrolledIn ?c } GROUP BY ?c }
  # separate subquery: an AVG over OPTIONAL (partly unbound) values would fail for the whole group
  { SELECT ?c (COUNT(?a) AS ?assessments) (ROUND(AVG(?score)) AS ?avg_score)
    WHERE { ?c edu:covers ?t . ?s edu:enrolledIn ?c . ?a edu:student ?s ; edu:topic ?t ; edu:score ?score }
    GROUP BY ?c }
}
ORDER BY ?course



,course,title,students,assessments,avg_score
0,DB150,Databases and Knowledge Graphs,24,87,66.0
1,DS101,Programming for Data Science,17,94,56.0
2,MA120,Mathematics for Data Science,22,98,62.0
3,ML201,Machine Learning,14,61,63.0


### Which topics are hardest?

Average scores (SIS) next to each topic's level and number of prerequisites (curriculum, with `edu:requires+` from Chapter 6):

In [9]:
sparql.run_query(conn, 'edugraph/topic_difficulty', graphs=ALL).head(8)

,topic,level,prerequisites,assessments,avg_score
0,Data visualization,2,8,11,45.0
1,NumPy,2,6,15,46.0
2,pandas,2,7,13,51.0
3,Neural networks,3,13,13,57.0
4,Statistics,2,2,18,57.0
5,Probability,2,1,20,58.0
6,Data structures,2,3,13,60.0
7,Control flow,1,1,14,60.0


The hardest topics sit **deep** in the prerequisite chains: *Data visualization* depends on 8 other topics. A weakness early in a chain shows up later. That's the next question.

### Prerequisite gaps

> *For every student: which topics did they score **below 50** on, where that topic is a prerequisite (at any depth) of something in a course they're **enrolled** in?*

That needs enrollments (SIS), course contents and the prerequisite chain (curriculum), and scores (SIS), all at once:

In [10]:
print(sparql.load_query('edugraph/prerequisite_gaps'))
gaps = sparql.run_query(conn, 'edugraph/prerequisite_gaps', graphs=ALL, threshold=50)
print(f'{len(gaps)} gaps for {gaps.student_name.nunique()} students')
gaps.head(10)

# Weak prerequisites: a student scored below ?threshold on a topic that a topic in one of their courses
# depends on (at any depth). Bind ?student to one student IRI, or leave it unbound for everyone.
# Parameters: ?threshold (integer, e.g. 50), optional ?student (IRI)
PREFIX edu: <http://example.org/edu#>

SELECT ?student_name ?weak_topic ?score ?course
       (GROUP_CONCAT(DISTINCT ?blocked_name; separator=", ") AS ?blocks)
WHERE {
  ?student edu:enrolledIn ?c ; edu:name ?student_name .
  ?c edu:code ?course ; edu:covers ?blocked .
  ?blocked edu:requires+ ?prerequisite ; edu:name ?blocked_name .
  ?assessment edu:student ?student ; edu:topic ?prerequisite ; edu:score ?score .
  FILTER(?score < ?threshold)
  ?prerequisite edu:name ?weak_topic .
}
GROUP BY ?student_name ?weak_topic ?score ?course
ORDER BY ?student_name ?score



67 gaps for 18 students


,student_name,weak_topic,score,course,blocks
0,Aarav Sharma,Statistics,33,ML201,"Linear regression, Model evaluation, Neural networks, Classification"
1,Ananya Singh,Statistics,39,ML201,"Linear regression, Model evaluation, Neural networks, Classification"
2,Ananya Singh,RDF and SPARQL,45,DB150,Knowledge graphs
3,Chen Tanaka,Probability,21,MA120,Statistics
4,Chen Tanaka,Algebra refresher,42,MA120,"Linear algebra, Calculus basics, Probability, Statistics"
5,Dara Mueller,NumPy,19,DS101,"pandas, Data visualization"
6,Dara Mueller,NumPy,19,ML201,"Linear regression, Model evaluation, Clustering, Neural networks, Classifica..."
7,Dara Mueller,Data structures,47,ML201,"Linear regression, Model evaluation, Clustering, Neural networks, Classifica..."
8,Dara Mueller,Data structures,47,DS101,"pandas, Data visualization, NumPy"
9,Dara Mueller,pandas,49,ML201,"Linear regression, Model evaluation, Clustering, Neural networks, Classifica..."


Read the first rows as advice: a low **Statistics** score blocks four Machine Learning topics, so revise Statistics **before** Linear regression.

The query has an optional **`?student`** parameter. Left unbound it covers everyone; bound, it gives one student's report. Chapter 13 turns exactly this into a REST endpoint `/students/{id}/gaps`.

In [11]:
one = gaps.student_name.iloc[0]
student_iri = sparql.run_query(conn, 'PREFIX edu: <http://example.org/edu#> SELECT ?s { ?s edu:name ?n }',
                               graphs=SIS['students'], n=one).s[0]
print(one, '→', student_iri)
sparql.run_query(conn, 'edugraph/prerequisite_gaps', graphs=ALL, threshold=50, student=URIRef(student_iri))

Aarav Sharma → http://example.org/edu/student/S023


,student_name,weak_topic,score,course,blocks
0,Aarav Sharma,Statistics,33,ML201,"Classification, Linear regression, Neural networks, Model evaluation"


## 10.6 Keeping imports fresh

An import is a **snapshot**. When the SIS changes, re-run it; `import_file` replaces the graph. A tiny change to the CSV, re-imported:

In [12]:
import shutil, tempfile

changed = Path(tempfile.mkdtemp()) / 'students.csv'
shutil.copy(EDU / 'sis' / 'students.csv', changed)
with open(changed, 'a', encoding='utf-8') as f:
    f.write('S041,Ada,Lovelace,ada.s041@student.example.edu,London,1,2025-09-01\n')

print('before:', imports.import_file(EDU / 'sis' / 'students.csv', MAPPINGS / 'students.sms', SIS['students']), 'triples')
print('after: ', imports.import_file(changed, MAPPINGS / 'students.sms', SIS['students']), 'triples  (+7 for the new student)')
_ = imports.import_file(EDU / 'sis' / 'students.csv', MAPPINGS / 'students.sms', SIS['students'])   # back to the original

before: 280 triples


after:  287 triples  (+7 for the new student)


Snapshots are fine for data that changes daily or weekly. When questions need **up-to-the-minute** data, virtualize instead.

## 10.7 Live virtual graphs over a database

With a database reachable from Stardog Cloud (a managed PostgreSQL such as Neon, Supabase, Amazon RDS, …), the same students can be queried **live**. The steps:

1. **Create a data source**: the connection.
2. **Create a virtual graph**: data source + mapping. `ontology/mappings/sis_students_sql.sms` is `students.sms` with `FROM SQL { SELECT … FROM students }` instead of `FROM CSV {}`.
3. **Query** it with `GRAPH <virtual://sis_students> { … }`, joined with ordinary graphs.

In the database, the table would be:

```sql
CREATE TABLE students (student_id text PRIMARY KEY, first_name text, last_name text, email text,
                       city text, year integer, enrolled_on date);
-- then load data/edugraph/sis/students.csv, e.g. with psql:  \copy students FROM 'students.csv' CSV HEADER
```

To run the cell below, add these to `.env`: `SIS_JDBC_URL` (e.g. `jdbc:postgresql://host:5432/dbname?sslmode=require`), `SIS_JDBC_USER`, `SIS_JDBC_PASSWORD`. Without them, it only explains.

In [13]:
import os
from stardog.content import MappingFile

LIVE = all(os.environ.get(k) for k in ('SIS_JDBC_URL', 'SIS_JDBC_USER', 'SIS_JDBC_PASSWORD'))
DATASOURCE, VIRTUAL_GRAPH = 'tutorial_sis', 'tutorial_sis_students'

if not LIVE:
    print('No SIS_JDBC_* settings in .env: skipping the live virtual graph (see the explanation below).')
else:
    with client.admin() as admin:
        try:
            admin.new_datasource(DATASOURCE, {
                'jdbc.url': os.environ['SIS_JDBC_URL'], 'jdbc.username': os.environ['SIS_JDBC_USER'],
                'jdbc.password': os.environ['SIS_JDBC_PASSWORD'], 'jdbc.driver': 'org.postgresql.Driver'})
            admin.new_virtual_graph(VIRTUAL_GRAPH, mappings=MappingFile(str(MAPPINGS / 'sis_students_sql.sms'), 'SMS2'),
                                    datasource=DATASOURCE, db=client.load_settings().database)
            LIVE_QUERY = f'''PREFIX edu: <http://example.org/edu#>
            SELECT ?name ?course {{
              GRAPH <virtual://{VIRTUAL_GRAPH}> {{ ?s edu:name ?name ; edu:city "Berlin" }}   # live, from the database
              ?s edu:enrolledIn/edu:code ?course                                            # stored in the graph
            }}'''
            display(sparql.run_query(conn, LIVE_QUERY, graphs=ALL))
            print(reasoning.query_plan(conn, LIVE_QUERY, ALL))       # look for the SQL pushed down to the database
        finally:
            for remove in (lambda: admin.virtual_graph(VIRTUAL_GRAPH).delete(), lambda: admin.datasource(DATASOURCE).delete()):
                try:
                    remove()
                except stardog.exceptions.StardogException:
                    pass

No SIS_JDBC_* settings in .env: skipping the live virtual graph (see the explanation below).


### What happens at query time: pushdown

When a query touches `GRAPH <virtual://…>`, Stardog **translates** that part into the source's language. Here the translation would be SQL: `SELECT … FROM students WHERE city = 'Berlin'`. The **filter** and, when possible, joins, ordering and limits are **pushed down**, so the database does the heavy lifting and only matching rows travel over the network. The query plan shows a virtual-graph node containing the generated SQL.

Write virtual-graph queries so they can be pushed down:

- Put **selective filters** on virtual data (`edu:city "Berlin"`), not after a big join with local data.
- Avoid functions the source can't translate; they force Stardog to fetch more rows and filter locally.
- Check the **plan**, and compare the SQL with what you'd write by hand.

### Virtualize or materialize?

| Situation | Choose |
|---|---|
| Files (CSV, JSON, spreadsheets) | **Materialize** (import), as in this chapter |
| Data changes constantly, and answers must be current | **Virtualize** |
| Huge source, and you need only small, filtered slices | **Virtualize** |
| Heavy analytics, reasoning, graph algorithms, many joins | **Materialize** |
| Source can't take extra query load, or is often offline | **Materialize** |
| Regulatory rule: data must stay in the source system | **Virtualize** |

The common architecture is a **hybrid**: the curriculum and other slow-changing reference data are materialized, the fast-moving operational data is virtualized, and a single SPARQL query spans both. Stardog can also **cache** virtual graphs (cache targets), and turn a virtual graph into stored triples with `admin.materialize_virtual_graph()`.

## 10.8 Exercises

**Exercise 1.** Which city has the most students with at least one prerequisite gap?

<details><summary>Solution</summary>

```python
gaps_all = sparql.run_query(conn, 'edugraph/prerequisite_gaps', graphs=ALL, threshold=50)
cities = sparql.run_query(conn, 'PREFIX edu: <http://example.org/edu#> SELECT ?student_name ?city { ?s edu:name ?student_name ; edu:city ?city }',
                          graphs=SIS['students'])
gaps_all.merge(cities).groupby('city').student_name.nunique().sort_values(ascending=False)
```
</details>

**Exercise 2.** For privacy, the analytics team must **not** see e-mail addresses. Make a copy of `students.sms` without the `edu:email` line, import it into `f'{PREFIX}:sis:students-anon'`, and check there are 240 triples (40 × 6).

<details><summary>Solution</summary>

```python
anon = Path(tempfile.mkdtemp()) / 'students_anon.sms'
anon.write_text('\n'.join(l for l in (MAPPINGS / 'students.sms').read_text().splitlines() if 'edu:email' not in l))
print(imports.import_file(EDU / 'sis' / 'students.csv', anon, f'{PREFIX}:sis:students-anon'))
```
Mappings are a good place to enforce data minimization: what isn't mapped never reaches the graph.
</details>

**Exercise 3.** List each student's **three weakest topics** (lowest scores) with the topic's level from the curriculum, for student `S007`.

<details><summary>Solution</summary>

```python
sparql.run_query(conn, '''PREFIX edu: <http://example.org/edu#>
SELECT ?topic ?level ?score { ?a edu:student ?s ; edu:topic ?t ; edu:score ?score . ?t edu:name ?topic ; edu:level ?level }
ORDER BY ?score LIMIT 3''', graphs=ALL, s=URIRef('http://example.org/edu/student/S007'))
```
</details>

**Exercise 4.** An online shop wants product data from its PostgreSQL database (5 million rows, prices change every minute) in the knowledge graph, and a monthly sales report over five years of order history. Virtualize or materialize, for each?

<details><summary>Solution</summary>

**Live prices: virtualize.** They change constantly, and queries usually need a few products, so pushdown keeps them fast. **Five years of order history for a monthly report: materialize** (or at least cache). Heavy aggregation over history shouldn't hit the operational database on every run.
</details>

In [14]:
# your answers here

## 10.9 Clean up

In [15]:
CLEAN_UP = True

if CLEAN_UP:
    graphs = sparql.run_query(conn, f'SELECT DISTINCT ?g {{ GRAPH ?g {{ ?s ?p ?o }} FILTER(STRSTARTS(STR(?g), "{PREFIX}")) }}').g.tolist()
    with client.transaction(conn):
        for g in graphs:
            conn.clear(graph_uri=g)
    print('Removed:', *graphs, sep='\n  ')
conn.close()

Removed:
  urn:tutorial:ch10:curriculum
  urn:tutorial:ch10:sis:students
  urn:tutorial:ch10:sis:enrollments
  urn:tutorial:ch10:sis:assessments


## Summary

- Knowledge is spread across systems. Stardog **materializes** (imports) or **virtualizes** (queries in place) them with the **same mappings**.
- A **data source** is a connection; a **virtual graph** is a data source + a mapping, queried as `GRAPH <virtual://name>`; an **import** applies a mapping once and stores the triples.
- **SMS2** mappings: `FROM` (CSV, JSON, SQL, …), `TO` (a triple template), `WHERE` (templates for IRIs, casts, string functions). **IRI templates must match the rest of the graph**, or nothing joins.
- Imported CSVs + a hand-written curriculum answered **prerequisite gaps**, a question neither source could answer alone.
- Live virtual graphs **push** filters and joins **down** to the source. Choose virtualize vs materialize per dataset; **hybrid** is the usual answer.

**Next: Chapter 11, Unstructured data and entity linking.** Extract topics from lesson documents with a local LLM (Ollama), link them to curriculum topics, and record where every extracted fact came from.